In [1]:
!pip install -q gdown

import gdown
import os
import zipfile
import shutil
import random

file_id = "1PwZXse2huPI_weaKCSq21Olu2ButPFyM"
output = "/kaggle/working/data.zip"

gdown.download(f"https://drive.google.com/uc?id={file_id}", output, quiet=False)

print("Download completed")



extract_path = "/kaggle/working/data"
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(output, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed")


paths_to_delete = [
    "/kaggle/working/data/.config",
    "/kaggle/working/data/.ipynb_checkpoints",
    "/kaggle/working/data/all_folders.zip",
    "/kaggle/working/data/drive",
    "/kaggle/working/.virtual_documents"
]

for path in paths_to_delete:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    elif os.path.isfile(path):
        os.remove(path)
        print(f"Deleted file: {path}")
    else:
        print(f"Path not found (skipped): {path}")


def split_classes_train_val(
    data_dir,
    train_ratio=0.8,
    seed=42,
    extensions=None
):
    """
    Splits each class folder into train/ and validation/ subfolders.

    Structure BEFORE:
    data/
        class1/
            img1.jpg
            img2.jpg
        class2/
            img1.jpg
            img2.jpg

    Structure AFTER:
    data/
        class1/
            train/
            validation/
        class2/
            train/
            validation/
    """

    random.seed(seed)

    classes = [
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ]

    for cls in classes:
        class_path = os.path.join(data_dir, cls)

        files = [
            f for f in os.listdir(class_path)
            if os.path.isfile(os.path.join(class_path, f))
        ]

        if extensions:
            files = [f for f in files if f.lower().endswith(extensions)]

        if len(files) == 0:
            continue

        random.shuffle(files)

        split_idx = int(len(files) * train_ratio)
        train_files = files[:split_idx]
        val_files = files[split_idx:]

        train_dir = os.path.join(class_path, "train")
        val_dir = os.path.join(class_path, "validation")

        os.makedirs(train_dir, exist_ok=True)
        os.makedirs(val_dir, exist_ok=True)

        for f in train_files:
            shutil.move(
                os.path.join(class_path, f),
                os.path.join(train_dir, f)
            )

        for f in val_files:
            shutil.move(
                os.path.join(class_path, f),
                os.path.join(val_dir, f)
            )

        print(
            f"{cls}: "
            f"{len(train_files)} train, "
            f"{len(val_files)} validation"
        )
data_dir = "/kaggle/working/data"

IMAGE_EXTENSIONS = (
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp",
    ".ppm", ".pgm", ".pbm"
)

split_classes_train_val(
    data_dir=data_dir,
    train_ratio=0.8,
    extensions=IMAGE_EXTENSIONS
)



def keep_only_train_val(data_dir):
    allowed = {"train", "validation"}

    classes = [
        d for d in os.listdir(data_dir)
        if os.path.isdir(os.path.join(data_dir, d))
    ]

    for cls in classes:
        class_path = os.path.join(data_dir, cls)

        for item in os.listdir(class_path):
            item_path = os.path.join(class_path, item)

            if os.path.isdir(item_path) and item not in allowed:
                shutil.rmtree(item_path)
                print(f"Deleted: {item_path}")

data_dir = "/kaggle/working/data"
keep_only_train_val(data_dir)

Downloading...
From (original): https://drive.google.com/uc?id=1PwZXse2huPI_weaKCSq21Olu2ButPFyM
From (redirected): https://drive.google.com/uc?id=1PwZXse2huPI_weaKCSq21Olu2ButPFyM&confirm=t&uuid=bdbe2e05-e723-46c2-b464-2de936993822
To: /kaggle/working/data.zip
100%|██████████| 993M/993M [00:09<00:00, 102MB/s]  


Download completed
Extraction completed
Deleted folder: /kaggle/working/data/.config
Deleted folder: /kaggle/working/data/.ipynb_checkpoints
Deleted file: /kaggle/working/data/all_folders.zip
Deleted folder: /kaggle/working/data/drive
Deleted folder: /kaggle/working/.virtual_documents
Hand Blower_dataset: 800 train, 200 validation
microwave_dataset: 800 train, 200 validation
Frying pan_dataset: 800 train, 200 validation
Vacuum_cleaner_dataset: 800 train, 200 validation
Chiffonier_dataset: 800 train, 200 validation
Waffle iron_dataset: 800 train, 200 validation
Teapot_dataset: 800 train, 200 validation
Toaster_dataset: 800 train, 200 validation
Dishwasher_dataset: 800 train, 200 validation
chair_dataset: 800 train, 200 validation
Bed_dataset: 755 train, 189 validation
stove_dataset: 800 train, 200 validation
wardrobe_dataset: 800 train, 200 validation
cleaver_dataset: 800 train, 200 validation
carpet_dataset: 800 train, 200 validation
Sofas_dataset: 800 train, 200 validation
refrigerato

In [2]:
import os
import shutil
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model


2025-12-23 05:44:30.671472: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766468671.143761      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766468671.276151      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1766468672.579972      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766468672.580011      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1766468672.580014      55 computation_placer.cc:177] computation placer alr

In [3]:
base = "/kaggle/working/data"
train_base = "/kaggle/working/data_train"
val_base = "/kaggle/working/data_val"

os.makedirs(train_base, exist_ok=True)
os.makedirs(val_base, exist_ok=True)

for cls in os.listdir(base):
    shutil.move(os.path.join(base, cls, "train"),
                os.path.join(train_base, cls))
    shutil.move(os.path.join(base, cls, "validation"),
                os.path.join(val_base, cls))


In [4]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
).flow_from_directory(
    "/kaggle/working/data_train",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)

val_gen = ImageDataGenerator(
    rescale=1./255
).flow_from_directory(
    "/kaggle/working/data_val",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical"
)


Found 19955 images belonging to 25 classes.
Found 4989 images belonging to 25 classes.


In [5]:
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False


I0000 00:00:1766468692.696645      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1766468692.700687      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [6]:
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model

x = base_model.output
x = Flatten()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.5)(x)
output = Dense(train_gen.num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)


In [7]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [8]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,             # لو 3 Epochs متحسنش val_loss يوقف
    restore_best_weights=True
)


In [9]:
history_initial = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=[early_stop]
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


I0000 00:00:1766468698.068665     142 service.cc:152] XLA service 0x7d864000d2a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1766468698.068702     142 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1766468698.068706     142 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1766468698.698941     142 cuda_dnn.cc:529] Loaded cuDNN version 91002


  1/624 ━━━━━━━━━━━━━━━━━━━━ 2:51:03 16s/step - accuracy: 0.0938 - loss: 3.7233

I0000 00:00:1766468712.623320     142 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


624/624 ━━━━━━━━━━━━━━━━━━━━ 300s 456ms/step - accuracy: 0.1829 - loss: 2.8907 - val_accuracy: 0.4967 - val_loss: 1.7303
Epoch 2/15
624/624 ━━━━━━━━━━━━━━━━━━━━ 254s 407ms/step - accuracy: 0.3074 - loss: 2.2098 - val_accuracy: 0.5582 - val_loss: 1.5663
Epoch 3/15
624/624 ━━━━━━━━━━━━━━━━━━━━ 256s 410ms/step - accuracy: 0.3429 - loss: 2.0721 - val_accuracy: 0.5891 - val_loss: 1.4116
Epoch 4/15
624/624 ━━━━━━━━━━━━━━━━━━━━ 255s 408ms/step - accuracy: 0.3600 - loss: 2.0107 - val_accuracy: 0.6138 - val_loss: 1.3769
Epoch 5/15
624/624 ━━━━━━━━━━━━━━━━━━━━ 255s 408ms/step - accuracy: 0.3764 - loss: 1.9583 - val_accuracy: 0.5879 - val_loss: 1.3581
Epoch 6/15
624/624 ━━━━━━━━━━━━━━━━━━━━ 252s 403ms/step - accuracy: 0.3770 - loss: 1.9421 - val_accuracy: 0.6158 - val_loss: 1.3016
Epoch 7/15
624/624 ━━━━━━━━━━━━━━━━━━━━ 257s 412ms/step - accuracy: 0.3904 - loss: 1.9110 - val_accuracy: 0.5973 - val_loss: 1.3635
Epoch 8/15
624/624 ━━━━━━━━━━━━━━━━━━━━ 258s 414ms/step - accuracy: 0.3981 - loss: 1.85

In [10]:

for layer in base_model.layers[-8:]:
    layer.trainable = True

from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(1e-5),  # learning rate صغير
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=25,
    callbacks=[early_stop]
)


Epoch 1/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 297s 449ms/step - accuracy: 0.4060 - loss: 1.8340 - val_accuracy: 0.6536 - val_loss: 1.1700
Epoch 2/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 276s 441ms/step - accuracy: 0.4391 - loss: 1.7022 - val_accuracy: 0.6827 - val_loss: 1.1059
Epoch 3/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 267s 428ms/step - accuracy: 0.4530 - loss: 1.6407 - val_accuracy: 0.6975 - val_loss: 1.0561
Epoch 4/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 265s 424ms/step - accuracy: 0.4864 - loss: 1.5534 - val_accuracy: 0.7232 - val_loss: 1.0149
Epoch 5/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 268s 428ms/step - accuracy: 0.5152 - loss: 1.4613 - val_accuracy: 0.7376 - val_loss: 0.9286
Epoch 6/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 265s 424ms/step - accuracy: 0.5447 - loss: 1.3680 - val_accuracy: 0.7529 - val_loss: 0.8777
Epoch 7/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 265s 425ms/step - accuracy: 0.5811 - loss: 1.2831 - val_accuracy: 0.7478 - val_loss: 0.8847
Epoch 8/25
624/624 ━━━━━━━━━━━━━━━━━━━━ 266s 426ms/step - accuracy: 0.6133 -

In [12]:
model.save("/kaggle/working/vgg16_finetuned_30epochs.h5")


In [14]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop_fine = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.3,
    patience=2,
    min_lr=1e-6
)


In [15]:
history_extra = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=7,
    callbacks=[early_stop_fine, reduce_lr]
)


Epoch 1/7
624/624 ━━━━━━━━━━━━━━━━━━━━ 264s 423ms/step - accuracy: 0.8590 - loss: 0.4839 - val_accuracy: 0.8423 - val_loss: 0.6711 - learning_rate: 1.0000e-05
Epoch 2/7
624/624 ━━━━━━━━━━━━━━━━━━━━ 269s 431ms/step - accuracy: 0.8665 - loss: 0.4519 - val_accuracy: 0.8435 - val_loss: 0.6585 - learning_rate: 1.0000e-05
Epoch 3/7
624/624 ━━━━━━━━━━━━━━━━━━━━ 265s 424ms/step - accuracy: 0.8704 - loss: 0.4331 - val_accuracy: 0.8400 - val_loss: 0.6799 - learning_rate: 1.0000e-05
Epoch 4/7
624/624 ━━━━━━━━━━━━━━━━━━━━ 265s 425ms/step - accuracy: 0.8745 - loss: 0.4147 - val_accuracy: 0.8356 - val_loss: 0.6880 - learning_rate: 1.0000e-05
Epoch 5/7
624/624 ━━━━━━━━━━━━━━━━━━━━ 263s 421ms/step - accuracy: 0.8912 - loss: 0.3628 - val_accuracy: 0.8517 - val_loss: 0.6305 - learning_rate: 3.0000e-06
Epoch 6/7
624/624 ━━━━━━━━━━━━━━━━━━━━ 264s 423ms/step - accuracy: 0.9027 - loss: 0.3195 - val_accuracy: 0.8493 - val_loss: 0.6463 - learning_rate: 3.0000e-06
Epoch 7/7
624/624 ━━━━━━━━━━━━━━━━━━━━ 263s 42

In [18]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

val_gen_noshuffle = ImageDataGenerator(
    rescale=1./255
).flow_from_directory(
    "/kaggle/working/data_val",
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False   
)


Found 4989 images belonging to 25 classes.


In [19]:
import numpy as np
from sklearn.metrics import classification_report

# Accuracy
loss, acc = model.evaluate(val_gen_noshuffle, verbose=1)
print(f"\nFinal Validation Accuracy: {acc:.4f}")

# Predictions
y_pred_probs = model.predict(val_gen_noshuffle, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# True labels
y_true = val_gen_noshuffle.classes

# Class names
class_names = list(val_gen_noshuffle.class_indices.keys())

print("\n====== Classification Report ======\n")
print(classification_report(y_true, y_pred, target_names=class_names))


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


156/156 ━━━━━━━━━━━━━━━━━━━━ 34s 217ms/step - accuracy: 0.8585 - loss: 0.6064

Final Validation Accuracy: 0.8517
156/156 ━━━━━━━━━━━━━━━━━━━━ 32s 203ms/step

====== Classification Report ======

                        precision    recall  f1-score   support

           Bed_dataset       0.75      0.87      0.80       189
    Chiffonier_dataset       0.91      0.92      0.92       200
     Crock pot_dataset       0.84      0.87      0.86       200
    Dishwasher_dataset       0.80      0.76      0.78       200
  Electric_fan_dataset       0.84      0.86      0.85       200
    Frying pan_dataset       0.90      0.82      0.86       200
   Hand Blower_dataset       0.76      0.81      0.78       200
   Mixing bowl_dataset       0.85      0.90      0.87       200
       Pitcher_dataset       0.84      0.85      0.85       200
   Salt Shaker_dataset       0.88      0.87      0.88       200
         Sofas_dataset       1.00      1.00      1.00       200
        Teapot_dataset       0.86   

In [21]:
model.save("vgg16_finetuned_best.keras")
